In [1]:
"""
Generates the DeepViG Grapher block architecture diagram.

Usage:
    python generate_architecture.py

Output:
    deepvig_architecture.png  (300 DPI, publication quality)
"""

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Rectangle

# ----------------------------------------------------------------
# Figure setup
# ----------------------------------------------------------------
fig, ax = plt.subplots(figsize=(18, 10))
ax.set_xlim(0, 18)
ax.set_ylim(0, 10)
ax.axis("off")

# ----------------------------------------------------------------
# Color palette (colorblind-safe + journal-friendly)
# ----------------------------------------------------------------
C_STD     = "#3A7DC9"   # Blue   — standard ops (LN, GCN)
C_POSTGN  = "#F39C12"   # Orange — PostGN  (novel)
C_LS      = "#27AE60"   # Green  — LayerScale (novel)
C_FFN     = "#8E44AD"   # Purple — FFN
C_RES     = "#7F8C8D"   # Grey   — residual connections
C_HL      = "#C00000"   # Red    — highlight for novel block

# ----------------------------------------------------------------
# Helper functions
# ----------------------------------------------------------------
def box(x, y, w, h, label, color, fs=13):
    """Draw a rounded rectangle with centered white text."""
    patch = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.05,rounding_size=0.12",
        facecolor=color, edgecolor="#1a1a1a", linewidth=2, zorder=5,
    )
    ax.add_patch(patch)
    ax.text(x + w / 2, y + h / 2, label,
            ha="center", va="center",
            fontsize=fs, color="white", weight="bold", zorder=6)


def arrow(x1, y1, x2, y2, color="#2C3E50", lw=2, dashed=False, zorder=4):
    """Draw an arrow from (x1,y1) to (x2,y2)."""
    ls = "--" if dashed else "-"
    a = FancyArrowPatch(
        (x1, y1), (x2, y2),
        arrowstyle="-|>", mutation_scale=20,
        color=color, linewidth=lw, linestyle=ls,
        shrinkA=0, shrinkB=0, zorder=zorder,
    )
    ax.add_patch(a)


def plus_op(x, y, r=0.24):
    """Draw a circled '+' operator."""
    ax.add_patch(plt.Circle((x, y), r, facecolor="white",
                            edgecolor="#1a1a1a", linewidth=2, zorder=10))
    ax.text(x, y, "+", ha="center", va="center",
            fontsize=20, weight="bold", zorder=11)


# ----------------------------------------------------------------
# Title
# ----------------------------------------------------------------
ax.text(9, 9.55, "DeepViG Grapher Block",
        ha="center", fontsize=22, weight="bold")

# ----------------------------------------------------------------
# Legend
# ----------------------------------------------------------------
legend_y = 8.85
items = [
    ("Standard Op",       C_STD),
    ("PostGN (Novel)",    C_POSTGN),
    ("LayerScale (Novel)", C_LS),
    ("FFN",               C_FFN),
]
for i, (label, color) in enumerate(items):
    x = 0.8 + i * 4.2
    ax.add_patch(Rectangle((x, legend_y), 0.4, 0.3,
                           facecolor=color, edgecolor="black", lw=1.5))
    ax.text(x + 0.55, legend_y + 0.15, label,
            fontsize=11, va="center", weight="bold")

# ================================================================
# SUBLAYER 1:  x -> LN1 -> GCN -> PostGN -> LayerScale(y1) -> (+)
# ================================================================
y_top = 6.3
h     = 0.8

ax.text(0.3, y_top + h / 2, r"$\mathbf{x}$",
        fontsize=20, weight="bold", va="center")

box(1.0, y_top, 1.4, h, "LN$_1$",    C_STD)
box(3.0, y_top, 1.6, h, "GCN (kNN)", C_STD)
box(5.1, y_top, 1.6, h, "PostGN",    C_POSTGN)
box(7.2, y_top, 2.0, h, r"LayerScale ($\gamma_1$)", C_LS)

plus_op(10.0, y_top + h / 2)
ax.text(11.0, y_top + h / 2, r"$\mathbf{x}'$",
        fontsize=20, weight="bold", va="center")

# Forward arrows
arrow(0.7,  y_top + h/2, 1.0,  y_top + h/2)   # x   -> LN1
arrow(2.4,  y_top + h/2, 3.0,  y_top + h/2)   # LN1 -> GCN
arrow(4.6,  y_top + h/2, 5.1,  y_top + h/2)   # GCN -> PostGN
arrow(6.7,  y_top + h/2, 7.2,  y_top + h/2)   # PostGN -> LayerScale
arrow(9.2,  y_top + h/2, 9.76, y_top + h/2)   # LayerScale -> +
arrow(10.24, y_top + h/2, 10.85, y_top + h/2) # + -> x'

# Residual connection (below the branch)
res_y = 5.2
arrow(0.7,  y_top,     0.7,  res_y, color=C_RES, lw=2.5)
arrow(0.7,  res_y,     10.0, res_y, color=C_RES, lw=2.5)
arrow(10.0, res_y,     10.0, y_top + h/2 - 0.24, color=C_RES, lw=2.5)
ax.text(5.35, res_y + 0.2, "Residual (identity)",
        fontsize=11, color=C_RES, weight="bold", style="italic")

# ================================================================
# SUBLAYER 2:  x' -> LN2 -> FFN -> LayerScale(y2) -> (+) -> x_out
# ================================================================
y_bot = 2.3

box(1.0, y_bot, 1.4, h, "LN$_2$",     C_STD)
box(3.0, y_bot, 1.6, h, "FFN (MLP)",  C_FFN)
box(5.1, y_bot, 2.0, h, r"LayerScale ($\gamma_2$)", C_LS)

plus_op(7.9, y_bot + h / 2)
ax.text(8.9, y_bot + h / 2, r"$\mathbf{x}_{\mathrm{out}}$",
        fontsize=20, weight="bold", va="center")

# Input from x' down to LN2
arrow(11.0, y_top + h/2 - 0.3, 11.0, 4.1, color="#2C3E50", lw=2)
arrow(11.0, 4.1,               1.7,  4.1, color="#2C3E50", lw=2)
arrow(1.7,  4.1,               1.7,  y_bot + h, color="#2C3E50", lw=2)

# Forward arrows
arrow(2.4, y_bot + h/2, 3.0, y_bot + h/2)  # LN2 -> FFN
arrow(4.6, y_bot + h/2, 5.1, y_bot + h/2)  # FFN -> LayerScale
arrow(7.1, y_bot + h/2, 7.66, y_bot + h/2) # LayerScale -> +
arrow(8.14, y_bot + h/2, 8.75, y_bot + h/2)# + -> x_out

# Residual for sublayer 2
res_y2 = 1.2
arrow(1.7, y_bot,   1.7,  res_y2, color=C_RES, lw=2.5)
arrow(1.7, res_y2,  7.9,  res_y2, color=C_RES, lw=2.5)
arrow(7.9, res_y2,  7.9,  y_bot + h/2 - 0.24, color=C_RES, lw=2.5)
ax.text(4.8, res_y2 + 0.2, "Residual (identity)",
        fontsize=11, color=C_RES, weight="bold", style="italic")

# ================================================================
# Highlights + annotation for novel components
# ================================================================
# Dashed highlight around PostGN + LayerScale (first branch)
ax.add_patch(Rectangle((4.9, y_top - 0.2), 4.5, h + 0.4,
                       facecolor="none", edgecolor=C_HL,
                       linewidth=2.5, linestyle="--", zorder=3))

# Dashed highlight around LayerScale (second branch)
ax.add_patch(Rectangle((4.9, y_bot - 0.2), 2.3, h + 0.4,
                       facecolor="none", edgecolor=C_HL,
                       linewidth=2.5, linestyle="--", zorder=3))

ax.annotate(
    "Novel components:\nPostGN + LayerScale",
    xy=(7.15, y_top + h + 0.2),
    xytext=(7.0, y_top + h + 1.1),
    fontsize=12, color=C_HL, weight="bold", ha="center",
    bbox=dict(boxstyle="round,pad=0.4", facecolor="#FFF4E6",
              edgecolor=C_HL, lw=2),
    arrowprops=dict(arrowstyle="->", color=C_HL, lw=2),
)

# ----------------------------------------------------------------
# Save
# ----------------------------------------------------------------
plt.tight_layout()
plt.savefig("deepvig_architecture.png", dpi=300, bbox_inches="tight",
            facecolor="white")
plt.close()
print("✓ Saved: deepvig_architecture.png")

✓ Saved: deepvig_architecture.png
